# *Environment Setup*

In [21]:
pip install requests beautifulsoup4 pandas lxml


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


# *Extract Data From a Single Page*

In [22]:
import requests
from bs4 import BeautifulSoup

url = "http://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
response.encoding = "utf-8"  # force correct encoding

soup = BeautifulSoup(response.text, "lxml")

books = soup.find_all("article", class_="product_pod")

for book in books[:5]:  # just first 5 to test
    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text
    availability = book.find("p", class_="instock availability").text.strip()
    rating = book.find("p", class_="star-rating")["class"][1]  # e.g. "Three"
    
    print(f"{title} | {price} | {availability} | {rating}")

A Light in the Attic | £51.77 | In stock | Three
Tipping the Velvet | £53.74 | In stock | One
Soumission | £50.10 | In stock | One
Sharp Objects | £47.82 | In stock | Four
Sapiens: A Brief History of Humankind | £54.23 | In stock | Five


# *Handle Pagination (Full Extract)*

In [23]:
import requests
from bs4 import BeautifulSoup
import time

all_books = []
page = 1

while True:
    url = f"http://books.toscrape.com/catalogue/page-{page}.html"
    response = requests.get(url)
    response.encoding = "utf-8"

    if response.status_code != 200:
        print(f"Stopped at page {page} (status {response.status_code})")
        break

    soup = BeautifulSoup(response.text, "lxml")
    books = soup.find_all("article", class_="product_pod")

    for book in books:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        availability = book.find("p", class_="instock availability").text.strip()
        rating = book.find("p", class_="star-rating")["class"][1]

        all_books.append({
            "title": title,
            "price": price,
            "availability": availability,
            "rating": rating
        })

    print(f"Scraped page {page} — {len(books)} books")
    page += 1
    time.sleep(1)  # politeness delay — don't hammer the server

print(f"\nTotal books scraped: {len(all_books)}")

Scraped page 1 — 20 books
Scraped page 2 — 20 books
Scraped page 3 — 20 books
Scraped page 4 — 20 books
Scraped page 5 — 20 books
Scraped page 6 — 20 books
Scraped page 7 — 20 books
Scraped page 8 — 20 books
Scraped page 9 — 20 books
Scraped page 10 — 20 books
Scraped page 11 — 20 books
Scraped page 12 — 20 books
Scraped page 13 — 20 books
Scraped page 14 — 20 books
Scraped page 15 — 20 books
Scraped page 16 — 20 books
Scraped page 17 — 20 books
Scraped page 18 — 20 books
Scraped page 19 — 20 books
Scraped page 20 — 20 books
Scraped page 21 — 20 books
Scraped page 22 — 20 books
Scraped page 23 — 20 books
Scraped page 24 — 20 books
Scraped page 25 — 20 books
Scraped page 26 — 20 books
Scraped page 27 — 20 books
Scraped page 28 — 20 books
Scraped page 29 — 20 books
Scraped page 30 — 20 books
Scraped page 31 — 20 books
Scraped page 32 — 20 books
Scraped page 33 — 20 books
Scraped page 34 — 20 books
Scraped page 35 — 20 books
Scraped page 36 — 20 books
Scraped page 37 — 20 books
Scraped pa

# *: Clean the Data with Pandas (Transform)*

In [24]:
import pandas as pd

df = pd.DataFrame(all_books)

# Price: strip £ symbol, convert to float
df["price"] = df["price"].str.replace("£", "", regex=False).astype(float)

# Availability: convert to a clean boolean instead of a sentence
df["in_stock"] = df["availability"].str.contains("In stock")
df = df.drop(columns=["availability"])

# Rating: convert word to number using a mapping dict
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df["rating"] = df["rating"].map(rating_map)

print(df.head())
print(df.info())

                                   title  price  rating  in_stock
0                   A Light in the Attic  51.77       3      True
1                     Tipping the Velvet  53.74       1      True
2                             Soumission  50.10       1      True
3                          Sharp Objects  47.82       4      True
4  Sapiens: A Brief History of Humankind  54.23       5      True
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   title     1000 non-null   object 
 1   price     1000 non-null   float64
 2   rating    1000 non-null   int64  
 3   in_stock  1000 non-null   bool   
dtypes: bool(1), float64(1), int64(1), object(1)
memory usage: 24.5+ KB
None


In [25]:
df.isnull().sum()

title       0
price       0
rating      0
in_stock    0
dtype: int64

# *Load into MySQL*

In [29]:

from sqlalchemy import create_engine

username = "root"
password = "0806"
host = "localhost"
database = "bookstore"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")

df.to_sql("scraped_books", con=engine, if_exists="replace", index=False)

print("Loaded into MySQL successfully.")

Loaded into MySQL successfully.
